# 101 - LangGraph: orquestación con estado

Este notebook introduce la idea principal de LangGraph: construir flujos con estado, decisiones y posibles reintentos.

Para que sea ejecutable en cualquier entorno, primero simulamos el patrón con Python estándar. Si `langgraph` está instalado, se muestra también el esquema equivalente.

## Preparación para Colab

La siguiente celda instala `langgraph` solo si el notebook se ejecuta en Google Colab y la librería no está disponible.

In [ ]:
import importlib.util
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))

def install_if_missing(import_name, package_name):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

if IN_COLAB:
    install_if_missing("langgraph", "langgraph")
    print("Dependencias de LangGraph preparadas.")
else:
    print("No se ha detectado Colab. Instalación automática omitida.")

## 1. Estado compartido

En LangGraph, los nodos intercambian un estado. Aquí usamos un diccionario con pregunta, contexto, respuesta, número de intentos y traza de ejecución.

In [ ]:
def new_state(question):
    return {
        "question": question,
        "context": [],
        "answer": "",
        "attempts": 0,
        "trace": [],
    }

state = new_state("¿Qué herramienta sirve para orquestar agentes con estado?")
print(state)

## 2. Nodos del grafo

Cada nodo es una función pequeña: recuperar contexto, generar respuesta, validar y decidir si hay que repetir.

In [ ]:
KNOWLEDGE = {
    "langgraph": "LangGraph permite construir flujos con estado, ciclos y decisiones.",
    "langchain": "LangChain permite componer prompts, modelos, herramientas y memoria.",
    "llamaindex": "LlamaIndex está especializado en RAG sobre documentos propios.",
}

def retrieve_node(state):
    text = state["question"].lower()
    context = [value for key, value in KNOWLEDGE.items() if key in text]
    if not context and state["attempts"] > 0:
        context = [KNOWLEDGE["langgraph"]]
    state["context"] = context
    state["trace"].append("retrieve")
    return state

def generate_node(state):
    state["attempts"] += 1
    if state["context"]:
        state["answer"] = f"Respuesta basada en contexto: {state['context'][0]}"
    else:
        state["answer"] = "No hay contexto suficiente."
    state["trace"].append("generate")
    return state

def validate_node(state):
    state["valid"] = bool(state["context"]) and "contexto" in state["answer"].lower()
    state["trace"].append("validate")
    return state

## 3. Ruta condicional con reintento

Si la validación falla, el flujo vuelve a recuperar contexto hasta un máximo de intentos. Este es el tipo de patrón donde LangGraph aporta valor frente a una cadena lineal.

In [ ]:
def run_graph(question, max_attempts=2):
    state = new_state(question)
    while True:
        state = retrieve_node(state)
        state = generate_node(state)
        state = validate_node(state)
        if state["valid"] or state["attempts"] >= max_attempts:
            state["trace"].append("end")
            return state
        state["trace"].append("retry")

print(run_graph("¿Qué es LangGraph?"))
print(run_graph("¿Qué herramienta usaría para agentes con estado?"))

## 4. Esquema equivalente con LangGraph real

La siguiente celda no es necesaria para entender el patrón, pero muestra cómo se expresaría con la librería real si está instalada.

In [ ]:
try:
    from langgraph.graph import StateGraph, END
    print("LangGraph disponible: se podría construir un StateGraph real.")
    print("Patrón: workflow = StateGraph(dict); add_node(...); add_conditional_edges(...); compile()")
except Exception as exc:
    print("LangGraph no está instalado. El ejemplo simulado anterior cubre el patrón principal.")
    print(type(exc).__name__, exc)

## Reto adicional

Añade un nodo `human_review_node` que marque la respuesta como aprobada o rechazada. Si se rechaza, el grafo debe repetir la generación una vez más.

Para profundizar: consulta `Documentacion/LangGraph_Documentacion.md`.